In [ ]:
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
import zipfile
import io

In [ ]:
def PreprocessDataset(SrcDir, DestDir, BucketsPath):
	SrcDir = Path(SrcDir)
	DestDir = Path(DestDir)
	
	DestDir.mkdir(parents=True, exist_ok=True)
	buckets = np.load(BucketsPath)
	nn = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(buckets)
	
	g = np.stack(np.meshgrid(np.arange(-128, 128), np.arange(-128, 128), indexing='ij'), -1).reshape(-1, 2)
	lut = nn.kneighbors(g)[1].reshape(256, 256).astype(np.uint16)
	
	with zipfile.ZipFile(DestDir / 'PrismPreprocessed.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
		
		for subset in ['train', 'val']:
			SrcLDir = SrcDir / subset / 'L'
			SrcABDir = SrcDir / subset / 'ab'

			LPaths = list(SrcLDir.glob('*.jpg'))
			ABPaths = list(SrcABDir.glob('*.npy'))
			
			for path in tqdm(ABPaths, mininterval=15.0):
				CenteredAB = np.load(path).astype(np.float32) - 128.0
				AIdx = (CenteredAB[..., 0] + 128.0).astype(int)
				BIdx = (CenteredAB[..., 1] + 128.0).astype(int)
				
				IdxMap = lut[AIdx, BIdx]
				
				buffer = io.BytesIO()
				np.save(buffer, IdxMap)
				
				ZipInternalPath = f"{subset}/ab/{path.name}"
				zipf.writestr(ZipInternalPath, buffer.getvalue())

			for path in tqdm(LPaths, mininterval=15.0):
				ZipInternalPath = f"{subset}/L/{path.name}"
				zipf.write(path, arcname=ZipInternalPath)

In [ ]:
if __name__ == '__main__':
	PreprocessDataset(
		SrcDir='/kaggle/input/datasets/pythonicvarun/colorization-dataset',
		DestDir='/kaggle/working/',
		BucketsPath='/kaggle/input/datasets/nkminion/prismtrain/CoordBuckets.npy'
	)